已知：

输入尺寸：通道数 C_in = 3，高 H = 32，宽 W = 32

卷积核数量：16，每个卷积核尺寸：3 × 5 × 5

填充 padding = 2，步幅 stride = 2

输出特征图的高度计算公式：
H_out = floor((H - kernel_size + 2 × padding) / stride) + 1
= floor((32 - 5 + 2 × 2) / 2) + 1
= floor((32 - 5 + 4) / 2) + 1
= floor(31 / 2) + 1
= 15 + 1
= 16

同理，宽度：W_out = 16

输出通道数 = 卷积核个数 = 16

因此，输出特征图尺寸：16 × 16 × 16

单个输出通道的一个像素值，计算过程：

每个卷积核的通道数等于输入通道数，即 3

卷积核尺寸：3 × 5 × 5

每个输出像素需要对所有输入通道的对应区域做点乘累加

点乘次数 = 卷积核所有元素的个数 = 3 × 5 × 5 = 75

因此，单个输出通道的一个像素值需要进行 75 次点乘操作。

In [1]:
import numpy as np

def max_pool2d(input, kernel_size, stride=None, padding=0):
    """
    二维最大池化的前向传播（手动实现，不使用底层 Pooling API）
    
    参数:
        input: numpy数组，形状为 (batch_size, channels, height, width)
        kernel_size: int 或 tuple，池化核大小
        stride: int 或 tuple，步幅，默认为 kernel_size
        padding: int 或 tuple，填充大小
    
    返回:
        output: 池化后的特征图
    """
    # 处理参数类型
    if isinstance(kernel_size, int):
        kernel_h = kernel_w = kernel_size
    else:
        kernel_h, kernel_w = kernel_size
    
    if stride is None:
        stride_h = stride_w = kernel_h
    elif isinstance(stride, int):
        stride_h = stride_w = stride
    else:
        stride_h, stride_w = stride
    
    if isinstance(padding, int):
        pad_h = pad_w = padding
    else:
        pad_h, pad_w = padding
    
    # 获取输入尺寸
    batch_size, channels, in_h, in_w = input.shape
    
    # 对输入进行填充
    padded_h = in_h + 2 * pad_h
    padded_w = in_w + 2 * pad_w
    padded_input = np.full((batch_size, channels, padded_h, padded_w), -np.inf)
    padded_input[:, :, pad_h:pad_h + in_h, pad_w:pad_w + in_w] = input
    
    # 计算输出尺寸
    out_h = (padded_h - kernel_h) // stride_h + 1
    out_w = (padded_w - kernel_w) // stride_w + 1
    
    # 初始化输出
    output = np.zeros((batch_size, channels, out_h, out_w))
    
    # 执行最大池化
    for b in range(batch_size):
        for c in range(channels):
            for i in range(out_h):
                for j in range(out_w):
                    # 计算窗口起始位置
                    start_h = i * stride_h
                    start_w = j * stride_w
                    # 提取窗口并取最大值
                    window = padded_input[b, c, 
                                          start_h:start_h + kernel_h, 
                                          start_w:start_w + kernel_w]
                    output[b, c, i, j] = np.max(window)
    
    return output


# 测试示例
if __name__ == "__main__":
    # 创建一个简单的测试输入 (batch=1, channel=1, height=4, width=4)
    x = np.array([[[[1, 2, 3, 4],
                    [5, 6, 7, 8],
                    [9, 10, 11, 12],
                    [13, 14, 15, 16]]]])
    
    print("输入:")
    print(x[0, 0])
    print()
    
    # 测试1: kernel=2, stride=2, padding=0
    out1 = max_pool2d(x, kernel_size=2, stride=2, padding=0)
    print("kernel=2, stride=2, padding=0 输出:")
    print(out1[0, 0])
    print()
    
    # 测试2: kernel=2, stride=1, padding=1
    out2 = max_pool2d(x, kernel_size=2, stride=1, padding=1)
    print("kernel=2, stride=1, padding=1 输出:")
    print(out2[0, 0])
    print()
    
    # 测试3: 多通道输入 (batch=1, channel=3, height=5, width=5)
    x_multi = np.random.randn(1, 3, 5, 5)
    out3 = max_pool2d(x_multi, kernel_size=3, stride=2, padding=1)
    print(f"多通道输入形状: {x_multi.shape}")
    print(f"多通道输出形状: {out3.shape}")

输入:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]]

kernel=2, stride=2, padding=0 输出:
[[ 6.  8.]
 [14. 16.]]

kernel=2, stride=1, padding=1 输出:
[[ 1.  2.  3.  4.  4.]
 [ 5.  6.  7.  8.  8.]
 [ 9. 10. 11. 12. 12.]
 [13. 14. 15. 16. 16.]
 [13. 14. 15. 16. 16.]]

多通道输入形状: (1, 3, 5, 5)
多通道输出形状: (1, 3, 3, 3)


一个5×5卷积层（不带偏置）的参数量：

输入通道数为C，输出通道数为C

卷积核尺寸：5×5

每个输出通道对应一个尺寸为C×5×5的卷积核

参数量 = C × C × 5 × 5 = 25 × C²

两个串联的3×3卷积层（不带偏置，两层通道数都为C）的总参数量：

第一层：输入通道C，输出通道C，卷积核3×3
参数量 = C × C × 3 × 3 = 9 × C²

第二层：输入通道C，输出通道C，卷积核3×3
参数量 = C × C × 3 × 3 = 9 × C²

总参数量 = 9 × C² + 9 × C² = 18 × C²

比较：18×C² < 25×C²，因此两个3×3卷积串联比一个5×5卷积的参数量更少。

In [2]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    NiN块：一个普通卷积层 + 两个1x1卷积层
    每层卷积后都紧跟ReLU激活函数
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(NiNBlock, self).__init__()
        
        self.block = nn.Sequential(
            # 普通卷积层（指定窗口大小、步幅、填充）
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, 
                      stride=stride, padding=padding),
            nn.ReLU(inplace=True),
            
            # 第一个1x1卷积层
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            
            # 第二个1x1卷积层
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.block(x)


# 使用torch.nn.Sequential直接定义的版本（作为备选）
def create_nin_block(in_channels, out_channels, kernel_size, stride=1, padding=0):
    """
    使用nn.Sequential直接创建NiN块
    """
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, 
                  stride=stride, padding=padding),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(inplace=True)
    )


# 测试示例
if __name__ == "__main__":
    # 测试NiNBlock类
    nin_block = NiNBlock(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=0)
    print("NiNBlock结构:")
    print(nin_block)
    print()
    
    # 测试forward
    x = torch.randn(1, 3, 224, 224)
    y = nin_block(x)
    print(f"输入形状: {x.shape}")
    print(f"输出形状: {y.shape}")
    print()
    
    # 测试函数式创建
    nin_block2 = create_nin_block(in_channels=96, out_channels=256, kernel_size=5, stride=1, padding=2)
    print("函数式创建NiNBlock:")
    print(nin_block2)

NiNBlock结构:
NiNBlock(
  (block): Sequential(
    (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
    (1): ReLU(inplace=True)
    (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU(inplace=True)
    (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU(inplace=True)
  )
)

输入形状: torch.Size([1, 3, 224, 224])
输出形状: torch.Size([1, 96, 54, 54])

函数式创建NiNBlock:
Sequential(
  (0): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (1): ReLU(inplace=True)
  (2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU(inplace=True)
  (4): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU(inplace=True)
)


已知：

批量数据：x₁ = 2, x₂ = 4, x₃ = 6, x₄ = 8

γ = 2, β = 1, ε = 0

批量归一化计算步骤：

计算均值 μ：
μ = (2 + 4 + 6 + 8) / 4 = 20 / 4 = 5

计算方差 σ²：
σ² = [(2-5)² + (4-5)² + (6-5)² + (8-5)²] / 4
= [(-3)² + (-1)² + (1)² + (3)²] / 4
= (9 + 1 + 1 + 9) / 4
= 20 / 4 = 5

计算标准化后的值（ε=0）：
ẑ₁ = (x₁ - μ) / √σ² = (2 - 5) / √5 = -3 / √5
ẑ₂ = (4 - 5) / √5 = -1 / √5
ẑ₃ = (6 - 5) / √5 = 1 / √5
ẑ₄ = (8 - 5) / √5 = 3 / √5

计算最终输出 y = γ·ẑ + β：
y₁ = 2 × (-3/√5) + 1 = -6/√5 + 1
y₂ = 2 × (-1/√5) + 1 = -2/√5 + 1
y₃ = 2 × (1/√5) + 1 = 2/√5 + 1
y₄ = 2 × (3/√5) + 1 = 6/√5 + 1

因此，最终输出值为：
y₁ = -6/√5 + 1, y₂ = -2/√5 + 1, y₃ = 2/√5 + 1, y₄ = 6/√5 + 1

In [3]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    残差块：两个3x3卷积层，每个卷积层后跟批量归一化层
    如果use_1x1conv=True，则使用1x1卷积调整输入以匹配输出形状
    """
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        
        # 第一个卷积层：3x3，步幅为stride（常用于降采样）
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 第二个卷积层：3x3，步幅为1
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                               stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 如果需要调整输入的通道数或尺寸，使用1x1卷积
        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                                      stride=stride)
        else:
            self.shortcut = None
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        # 残差路径
        residual = self.conv1(x)
        residual = self.bn1(residual)
        residual = self.relu(residual)
        
        residual = self.conv2(residual)
        residual = self.bn2(residual)
        
        # 快捷连接（恒等映射或1x1卷积调整）
        if self.shortcut is not None:
            identity = self.shortcut(x)
        else:
            identity = x
        
        # 相加后激活
        out = residual + identity
        out = self.relu(out)
        
        return out


# 测试示例
if __name__ == "__main__":
    # 测试1：输入输出通道相同，不使用1x1卷积
    block1 = Residual(in_channels=64, out_channels=64, use_1x1conv=False)
    x1 = torch.randn(2, 64, 32, 32)
    y1 = block1(x1)
    print(f"输入形状: {x1.shape}")
    print(f"输出形状: {y1.shape}")
    print(f"残差块结构（同通道）:")
    print(block1)
    print()
    
    # 测试2：输入输出通道不同，使用1x1卷积
    block2 = Residual(in_channels=32, out_channels=64, use_1x1conv=True, stride=2)
    x2 = torch.randn(2, 32, 64, 64)
    y2 = block2(x2)
    print(f"输入形状: {x2.shape}")
    print(f"输出形状: {y2.shape}")
    print(f"残差块结构（通道变化+降采样）:")
    print(block2)

输入形状: torch.Size([2, 64, 32, 32])
输出形状: torch.Size([2, 64, 32, 32])
残差块结构（同通道）:
Residual(
  (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
)

输入形状: torch.Size([2, 32, 64, 64])
输出形状: torch.Size([2, 64, 32, 32])
残差块结构（通道变化+降采样）:
Residual(
  (conv1): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (shortcut): Conv2d(32, 64, kernel_size=(1, 1), stride=(2, 2))
  (relu): ReLU(inplace=True)
)


为什么底层特征提取层用小学习率（或冻结），顶层输出层用大学习率？

底层特征提取层（靠近输入）学习的是通用的、与任务无关的低级特征（如边缘、纹理、形状等）。这些特征在大多数视觉任务中是共享的，源数据集（如ImageNet）已经学到了很好的表示，因此不需要大幅度调整。用小学习率或冻结可以保持这些通用特征，避免破坏已有知识，同时减少过拟合风险。

顶层输出层（靠近输出）学习的是与具体任务相关的高级语义特征和类别判别信息。由于目标数据集的任务类别可能与源数据集不同（例如ImageNet的1000类 vs 目标任务的猫狗二分类），这层通常需要重新初始化并从头训练。设置较大的学习率可以让这层快速适应新任务的数据分布。

目标数据集非常小且与源数据集非常相似时的微调策略：

冻结大部分底层特征提取层的参数，只微调最后几层（甚至只微调最后的全连接层）

使用较小的整体学习率，避免在小数据集上大幅度更新预训练权重

采用更强的正则化方法，如Dropout、权重衰减（L2正则化）

使用数据增广来扩充目标数据集

考虑使用更小的批量大小或早停（Early Stopping）

可以只重新训练最终输出层（分类器），保持所有特征提取层冻结

In [4]:
import torchvision.transforms as transforms
from torchvision.transforms import Compose, RandomResizedCrop, RandomHorizontalFlip, ColorJitter, ToTensor

def create_augmentation_pipeline():
    """
    创建图像增广管道（Pipeline）
    
    包含：
    1. 随机裁剪并缩放到224x224（面积比例0.08~1.0）
    2. 50%概率水平翻转
    3. 随机改变亮度、对比度、饱和度（变化幅度0.5）
    4. 转换为PyTorch张量
    """
    pipeline = Compose([
        # 1. 随机裁剪：面积比例0.08~1.0，缩放至224x224
        RandomResizedCrop(size=224, scale=(0.08, 1.0)),
        
        # 2. 50%概率水平翻转
        RandomHorizontalFlip(p=0.5),
        
        # 3. 随机改变亮度、对比度、饱和度，变化幅度0.5
        # brightness=0.5, contrast=0.5, saturation=0.5, hue=0
        ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
        
        # 4. 转换为PyTorch张量
        ToTensor()
    ])
    
    return pipeline


# 备选版本：带归一化的完整管道（实际训练常用）
def create_full_augmentation_pipeline(mean=[0.485, 0.456, 0.406], 
                                       std=[0.229, 0.224, 0.225]):
    """
    创建完整的图像增广管道（包含归一化）
    
    参数:
        mean: 各通道均值
        std: 各通道标准差
    """
    pipeline = Compose([
        RandomResizedCrop(size=224, scale=(0.08, 1.0)),
        RandomHorizontalFlip(p=0.5),
        ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
        ToTensor(),
        transforms.Normalize(mean=mean, std=std)  # 标准化，便于模型训练
    ])
    
    return pipeline


# 测试示例
if __name__ == "__main__":
    from PIL import Image
    import numpy as np
    
    # 创建增广管道
    aug_pipeline = create_augmentation_pipeline()
    
    print("图像增广管道结构:")
    print(aug_pipeline)
    print()
    
    # 测试：创建一张示例图像并应用增广
    # 创建一个假图像（3通道，300x300）
    dummy_image = Image.fromarray(np.random.randint(0, 255, (300, 300, 3), dtype=np.uint8))
    
    print(f"原始图像尺寸: {dummy_image.size}")
    
    # 应用增广
    augmented_tensor = aug_pipeline(dummy_image)
    
    print(f"增广后张量形状: {augmented_tensor.shape}")
    print(f"张量取值范围: [{augmented_tensor.min():.3f}, {augmented_tensor.max():.3f}]")
    
    # 显示完整管道（带归一化）
    full_pipeline = create_full_augmentation_pipeline()
    print("\n完整增广管道（带归一化）:")
    print(full_pipeline)

图像增广管道结构:
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)

原始图像尺寸: (300, 300)
增广后张量形状: torch.Size([3, 224, 224])
张量取值范围: [0.000, 0.749]

完整增广管道（带归一化）:
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


已知：

真实框 A = [10, 10, 50, 50]（左上角x, 左上角y, 右下角x, 右下角y）

预测框 B = [30, 30, 70, 70]

计算A和B的相交区域：

相交区域左上角x = max(10, 30) = 30

相交区域左上角y = max(10, 30) = 30

相交区域右下角x = min(50, 70) = 50

相交区域右下角y = min(50, 70) = 50

相交区域宽度 = 50 - 30 = 20
相交区域高度 = 50 - 30 = 20
相交面积 = 20 × 20 = 400

计算A的面积：
A宽度 = 50 - 10 = 40
A高度 = 50 - 10 = 40
A面积 = 40 × 40 = 1600

计算B的面积：
B宽度 = 70 - 30 = 40
B高度 = 70 - 30 = 40
B面积 = 40 × 40 = 1600

计算并集面积：
并集面积 = A面积 + B面积 - 相交面积
= 1600 + 1600 - 400 = 2800

计算IoU：
IoU = 相交面积 / 并集面积 = 400 / 2800 = 1 / 7 ≈ 0.142857

因此，边界框A和B之间的IoU准确值为 1/7（约0.142857）。

In [5]:
import torch
import torch.nn.functional as F
import numpy as np

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1, reduction='mean'):
    """
    计算标签平滑后的交叉熵损失
    
    参数:
        logits: 模型输出，形状 (batch_size, num_classes)，未经softmax
        labels: 真实标签，形状 (batch_size,)
        epsilon: 平滑因子，默认0.1
        reduction: 损失计算方式，'mean'、'sum'或'none'
    
    返回:
        损失值
    """
    batch_size, num_classes = logits.shape
    
    # 将logits转换为概率分布
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 创建平滑标签
    # 真实类别概率为 1 - epsilon，其他类别为 epsilon / (K-1)
    smooth_labels = torch.full_like(log_probs, epsilon / (num_classes - 1))
    smooth_labels.scatter_(1, labels.unsqueeze(1), 1 - epsilon)
    
    # 计算交叉熵损失
    loss = -torch.sum(smooth_labels * log_probs, dim=-1)
    
    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:
        return loss


# 备选版本：使用KL散度实现（原理相同）
def label_smoothing_cross_entropy_v2(logits, labels, epsilon=0.1, reduction='mean'):
    """
    使用KL散度实现标签平滑交叉熵损失
    """
    batch_size, num_classes = logits.shape
    
    # 计算log softmax
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 创建平滑目标分布
    target_dist = torch.full_like(log_probs, epsilon / (num_classes - 1))
    target_dist.scatter_(1, labels.unsqueeze(1), 1 - epsilon)
    
    # KL散度损失 = sum(target * (log(target) - log(probs)))
    # 等价于交叉熵
    loss = F.kl_div(log_probs, target_dist, reduction='none').sum(dim=-1)
    
    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:
        return loss


# 使用PyTorch内置模块的方式（自定义损失函数类）
class LabelSmoothingCrossEntropy(torch.nn.Module):
    """
    标签平滑交叉熵损失类
    """
    def __init__(self, epsilon=0.1, reduction='mean'):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.epsilon = epsilon
        self.reduction = reduction
    
    def forward(self, logits, labels):
        return label_smoothing_cross_entropy(logits, labels, self.epsilon, self.reduction)


# 测试示例
if __name__ == "__main__":
    # 设置随机种子
    torch.manual_seed(42)
    
    # 模拟数据：batch_size=4, num_classes=10
    batch_size = 4
    num_classes = 10
    logits = torch.randn(batch_size, num_classes)
    labels = torch.tensor([0, 2, 5, 7])
    
    print("模型输出(logits):")
    print(logits)
    print(f"\n真实标签: {labels}")
    print()
    
    # 普通交叉熵损失（作为对比）
    ce_loss = F.cross_entropy(logits, labels, reduction='mean')
    print(f"普通交叉熵损失: {ce_loss.item():.4f}")
    
    # 标签平滑交叉熵损失
    ls_loss = label_smoothing_cross_entropy(logits, labels, epsilon=0.1)
    print(f"标签平滑交叉熵损失 (epsilon=0.1): {ls_loss.item():.4f}")
    
    # 不同epsilon值的效果
    for eps in [0.0, 0.05, 0.1, 0.2]:
        loss = label_smoothing_cross_entropy(logits, labels, epsilon=eps)
        print(f"  epsilon={eps}: loss={loss.item():.4f}")
    
    # 测试reduction选项
    print("\n测试reduction选项:")
    loss_none = label_smoothing_cross_entropy(logits, labels, epsilon=0.1, reduction='none')
    loss_sum = label_smoothing_cross_entropy(logits, labels, epsilon=0.1, reduction='sum')
    print(f"reduction='none': {loss_none}")
    print(f"reduction='sum': {loss_sum.item():.4f}")
    
    # 测试损失函数类
    print("\n测试损失函数类:")
    criterion = LabelSmoothingCrossEntropy(epsilon=0.1)
    loss_class = criterion(logits, labels)
    print(f"类方式计算损失: {loss_class.item():.4f}")

模型输出(logits):
tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047,
         -0.7521,  1.6487],
        [-0.3925, -1.4036, -0.7279, -0.5594, -0.7688,  0.7624,  1.6423, -0.1596,
         -0.4974,  0.4396],
        [-0.7581,  1.0783,  0.8008,  1.6806,  0.0349,  0.3211,  1.5736, -0.8455,
          1.3123,  0.6872],
        [-1.0892, -0.3553, -1.4181,  0.8963,  0.0499,  2.2667,  1.1790, -0.4345,
         -1.3864, -1.2862]])

真实标签: tensor([0, 2, 5, 7])

普通交叉熵损失: 2.6813
标签平滑交叉熵损失 (epsilon=0.1): 2.7016
  epsilon=0.0: loss=2.6813
  epsilon=0.05: loss=2.6915
  epsilon=0.1: loss=2.7016
  epsilon=0.2: loss=2.7219

测试reduction选项:
reduction='none': tensor([1.4113, 3.2120, 2.8427, 3.3404])
reduction='sum': 10.8065

测试损失函数类:
类方式计算损失: 2.7016
